In [ ]:
# Parameters — overridable via papermill -p
# (Defaults reproduce the existing BEADs-on-BEADs run.)
TRAIN_JSONL = "../../data/frozen/beads/sizes/full/train.jsonl"
TEST_JSONL = "../../data/frozen/beads/sizes/full/test.jsonl"
RUN_NAME = "tfidf_beads_full__on__beads"
OUTPUT_DIR = "."
SMOKE_TEST = False

In [ ]:
# Smoke-test override (papermill injects above this cell so the conditional
# sees the overridden SMOKE_TEST value, not the default).
if SMOKE_TEST:
    pass  # nothing scale-dependent here; TF-IDF runs fast at any size

In [ ]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)
import time

In [ ]:
train_texts, train_labels = [], []

with open(TRAIN_JSONL) as f:
    for line in f:
        row = json.loads(line)
        train_texts.append(row['text'])
        train_labels.append(row['label_int'])

print(f'Train rows: {len(train_texts)}')

In [ ]:
test_texts, test_labels, test_rows = [], [], []

with open(TEST_JSONL) as f:
    for line in f:
        row = json.loads(line)
        test_texts.append(row['text'])
        test_labels.append(row['label_int'])
        test_rows.append(row)

print(f'Test rows: {len(test_texts)}')

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        random_state=42,
        C=1.0
    ))
])

start = time.time()
pipeline.fit(train_texts, train_labels)
elapsed = time.time() - start

print(f'Training complete in {elapsed:.2f} seconds')

In [ ]:
preds = pipeline.predict(test_texts)
print(f'Predictions complete. Sample: {preds[:10]}')

In [ ]:
accuracy = accuracy_score(test_labels, preds)
precision_pos = precision_score(test_labels, preds, pos_label=1)
recall_pos = recall_score(test_labels, preds, pos_label=1)
f1_pos = f1_score(test_labels, preds, pos_label=1)
f1_macro = f1_score(test_labels, preds, average='macro')
clf_report = classification_report(
    test_labels, preds,
    target_names=['non-biased', 'biased']
)

print(clf_report)
print(f'Accuracy:   {accuracy:.4f}')
print(f'F1 macro:   {f1_macro:.4f}')
print(f'F1 pos:     {f1_pos:.4f}')

In [ ]:
def _infer_dataset(p):
    parts = p.split("/")
    if "frozen" in parts:
        i = parts.index("frozen")
        if i+1 < len(parts) and not parts[i+1].endswith(".jsonl"):
            return parts[i+1]
    return None

eval_metrics = {
    "run_name": RUN_NAME,
    "train_dataset": _infer_dataset(TRAIN_JSONL),
    "eval_dataset": _infer_dataset(TEST_JSONL),
    "n_examples": len(test_labels),
    "accuracy": accuracy,
    "precision_pos": precision_pos,
    "recall_pos": recall_pos,
    "f1_pos": f1_pos,
    "f1_macro": f1_macro,
    "classification_report": clf_report,
    "eval_seconds": elapsed,
    "smoke_test": False,
    "adapter_path": None,
    "test_jsonl": TEST_JSONL
}

with open(f"{OUTPUT_DIR}/tfidf_eval_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=2)

print('Saved tfidf_eval_metrics.json')

In [ ]:
with open(f"{OUTPUT_DIR}/tfidf_predictions.jsonl", "w") as f:
    for row, pred in zip(test_rows, preds):
        out = {
            "text": row['text'],
            "label_int": row['label_int'],
            "label_str": row['label_str'],
            "pred_int": int(pred),
            "pred_str": "biased" if pred == 1 else "non-biased"
        }
        f.write(json.dumps(out) + '\n')

print('Saved tfidf_predictions.jsonl')